In [22]:
import os
os.chdir("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib

import yfinance as yf
from fredapi import Fred

import preprocessing
importlib.reload(preprocessing)
from preprocessing import dataset_summary

import data_splitting
importlib.reload(data_splitting)
# from data_splitting import

import models
importlib.reload(models)
# from models import

import utils
importlib.reload(utils)
# from utils import


<module 'utils' from 'c:\\Users\\pepob\\OneDrive\\Documentos\\ML\\Beresten_Salivaras_Proyecto_Final\\src\\utils.py'>

In [ ]:
sp500 = yf.download("^GSPC", start="2008-01-01", end="2024-12-31")
sp500.to_csv("../data/sp500_raw.csv")

fred = Fred(api_key="952ffd63d97d62b1fbf610042d229022")

vix = fred.get_series("VIXCLS",    observation_start="2008-01-01", observation_end="2024-12-31")
t10y2y = fred.get_series("T10Y2Y",   observation_start="2008-01-01", observation_end="2024-12-31")
fedfunds = fred.get_series("FEDFUNDS", observation_start="2008-01-01", observation_end="2024-12-31")
cpi = fred.get_series("CPIAUCSL", observation_start="2008-01-01", observation_end="2024-12-31")
unrate = fred.get_series("UNRATE",   observation_start="2008-01-01", observation_end="2024-12-31")

macro = pd.DataFrame({
    "vix": vix,
    "t10y2y": t10y2y,
    "fedfunds": fedfunds,
    "cpi": cpi,
    "unrate": unrate
})

macro.to_csv("../data/macro_fred.csv")

[*********************100%***********************]  1 of 1 completed


In [36]:
sp500 = pd.read_csv("../data/sp500_raw.csv", skiprows=3, header=None,
                    names=["Date", "Close", "High", "Low", "Open", "Volume"],
                    index_col=0, parse_dates=True)
macro = pd.read_csv("../data/macro_fred.csv", index_col=0, parse_dates=True)
news = pd.read_csv("../data/sp500_news.csv", parse_dates=["Date"])

dataset_summary("S&P 500 Prices", sp500)
dataset_summary("Macro FRED", macro)
dataset_summary("News Headlines", news)


  S&P 500 Prices  |  4,278 rows  x  5 columns


,dtype,non_null,null,null_%,unique
Close,float64,4278,0,0.0,4255
High,float64,4278,0,0.0,4242
Low,float64,4278,0,0.0,4250
Open,float64,4278,0,0.0,4246
Volume,int64,4278,0,0.0,4211



── Random sample (5 rows) ──


,Close,High,Low,Open,Volume
Date,,,,,
2010-03-02,1118.310059,1123.459961,1116.510010,1117.010010,4134680000
2009-08-18,989.669983,991.200012,980.619995,980.619995,4198970000
2018-02-26,2779.600098,2780.639893,2753.780029,2757.370117,3436590000
2013-02-04,1495.709961,1513.170044,1495.020020,1513.170044,3390000000
2010-05-18,1120.800049,1148.660034,1117.199951,1138.780029,6170840000



  Macro FRED  |  4,493 rows  x  5 columns


...

,vix,t10y2y,fedfunds,cpi,unrate
2008-08-05,21.14,1.50,NaN,NaN,NaN
2017-07-21,9.36,0.88,NaN,NaN,NaN
2011-11-11,30.04,NaN,NaN,NaN,NaN
2012-03-28,15.47,1.87,NaN,NaN,NaN
2010-10-07,21.56,2.05,NaN,NaN,NaN



  News Headlines  |  19,127 rows  x  3 columns


,dtype,non_null,null,null_%,unique
Title,object,19127,0,0.0,18054
Date,datetime64[ns],19127,0,0.0,3507
CP,float64,19127,0,0.0,3491



── Random sample (5 rows) ──


,Title,Date,CP
438,India: Land of Hope and Growth - MINING.COM,2010-11-29,1187.76
9393,Stock Market Crash? Here Is Why We Are In A Go...,2020-08-28,3508.01
385,"CNNMoney.com Market Report - Sep. 21, 2010",2010-09-21,1139.78
2177,15 Popular S&P 500 Valuation Metrics,2013-08-09,1691.42
15033,Western Digital stock leads S&P 500 gainers on...,2023-05-15,4136.28


In [ ]:
macro = macro.ffill() # Forward fill for monthly data to daily frequency
macro = macro.reset_index() # Makes 'Date' a column instead of index
macro = macro.rename(columns={"index": "Date"})
macro = macro.dropna()

## Datasets

Se utilizan tres fuentes de datos para el período 2008–2024:

**S&P 500 Prices** — Descargado via `yfinance` (ticker `^GSPC`). Contiene 4,279 días 
de trading con columnas Open, High, Low, Close y Volume. Se usa el precio de cierre 
ajustado para calcular el target y los indicadores técnicos.

**Macro FRED** — Descargado via `fredapi` desde la Federal Reserve Economic Data. 
Contiene 5 series: VIX y T10Y2Y (frecuencia diaria), y Fed Funds Rate, CPI y 
Desempleo (frecuencia mensual, rellenadas con forward fill para alinear con los días 
de trading). Total: 4,492 observaciones.

**News Headlines** — Dataset de Kaggle con 19,127 headlines financieros del S&P 500 
entre 2008 y 2024, con múltiples noticias por día. Se usa como input para FinBERT, 
que genera un score de sentiment diario agregado. Cubre 3,507 fechas únicas.

### Limpieza
Solo el dataset de FRED presentó una muestra con valores nulos, correspondiente
al primer día sin dato en VIX y T10Y2Y. Este fue eliminado con `dropna()`. 